In [2]:
%cd TRUST4

/home/pc11/TRUST4_workspace/TRUST4


In [2]:
# @markdown This cell compile the TRUST4 algorithm
!make
!echo "TRUST4 is ready to use!"

make: Nothing to be done for 'all'.
TRUST4 is ready to use!


In [3]:
!./run-trust4 -h

[Mon Jun 29 14:14:42 2026] TRUST4 v1.1.10-r633 begins.
Unknown parameter -h


In [4]:
!bash trust-example-test.sh

[Mon Jun 29 14:14:49 2026] TRUST4 v1.1.10-r633 begins.
[Mon Jun 29 14:14:49 2026] SYSTEM CALL: /home/pc11/TRUST4_workspace/TRUST4/bam-extractor -b example/example.bam -t 1 -f hg38_bcrtcr.fa -o example_test_toassemble 
[Mon Jun 29 14:14:49 2026] Start to extract candidate reads from bam file.
[Mon Jun 29 14:14:49 2026] Finish obtaining the candidate read ids.
[Mon Jun 29 14:14:49 2026] Finish extracting reads.
[Mon Jun 29 14:14:49 2026] SYSTEM CALL: /home/pc11/TRUST4_workspace/TRUST4/trust4  -f hg38_bcrtcr.fa -o example_test -1 example_test_toassemble_1.fq -2 example_test_toassemble_2.fq
[Mon Jun 29 14:14:49 2026] Start to assemble reads.
[Mon Jun 29 14:14:49 2026] Found 394 reads.
[Mon Jun 29 14:14:49 2026] Finish sorting the reads.
[Mon Jun 29 14:14:49 2026] Finish rough annotations.
[Mon Jun 29 14:14:49 2026] Assembled 394 reads.
[Mon Jun 29 14:14:49 2026] Try to rescue 0 reads for assembly.
[Mon Jun 29 14:14:49 2026] Rescued 0 reads.
[Mon Jun 29 14:14:49 2026] Extend assemblies by m

In [5]:
# @markdown Since this is a 10x single-cell dataset, R1 contains the cell barcode and UMI, while R2 contains the cDNA/transcript read. Therefore, TRUST4 is run as follows:

!./run-trust4 \
 -u "/home/pc11/TRUST4_workspace/SRR13342054_2.fastq" \
 -f human_IMGT+C.fa \
--ref human_IMGT+C.fa \
--barcode "/home/pc11/TRUST4_workspace/SRR13342054_1.fastq" \
--UMI "/home/pc11/TRUST4_workspace/SRR13342054_1.fastq" \
--readFormat bc:0:15,um:16:25 \
-t 4 \
-o "/home/pc11/TRUST4_workspace/SRR13342054_TRUST4"

[Mon Jun 29 14:15:45 2026] TRUST4 v1.1.10-r633 begins.
[Mon Jun 29 14:15:45 2026] SYSTEM CALL: /home/pc11/TRUST4_workspace/TRUST4/fastq-extractor -t 4 -f human_IMGT+C.fa -o /home/pc11/TRUST4_workspace/SRR13342054_TRUST4_toassemble  --readFormat bc:0:15,um:16:25 -u /home/pc11/TRUST4_workspace/SRR13342054_2.fastq --barcode /home/pc11/TRUST4_workspace/SRR13342054_1.fastq --UMI /home/pc11/TRUST4_workspace/SRR13342054_1.fastq
[Mon Jun 29 14:15:45 2026] Start to extract candidate reads from read files.
[Mon Jun 29 14:24:49 2026] Finish extracting reads.
[Mon Jun 29 14:24:49 2026] SYSTEM CALL: /home/pc11/TRUST4_workspace/TRUST4/trust4  -t 4 -f human_IMGT+C.fa -o /home/pc11/TRUST4_workspace/SRR13342054_TRUST4 -u /home/pc11/TRUST4_workspace/SRR13342054_TRUST4_toassemble.fq --barcode /home/pc11/TRUST4_workspace/SRR13342054_TRUST4_toassemble_bc.fa --UMI /home/pc11/TRUST4_workspace/SRR13342054_TRUST4_toassemble_umi.fa
[Mon Jun 29 14:24:50 2026] Start to assemble reads.
[Mon Jun 29 14:24:54 2026] F

In [2]:
!pip install pandas

/usr/bin/sh: 1: pip: not found


In [4]:

import pandas as pd

# Path to the TRUST4 barcode report file
trust_path = "/home/pc11/TRUST4_workspace/SRR13342054_TRUST4_barcode_report.tsv"

# Output CSV file path for paired clonotypes
out_path = "/home/pc11/TRUST4_workspace/trust4_paired_all_clonotypes.csv"

# Load the barcode report
trust = pd.read_csv(trust_path, sep="\t")

print(f"Total barcode rows in TRUST4 output: {trust.shape[0]}")


def parse_chain(chain_value, chain_type):
    """
    Parses the TRUST4 chain format:
    V,D,J,C,CDR3nt,CDR3aa,count,contig_id,score,flag

    chain1 = typically TRB (beta)
    chain2 = typically TRA (alpha)

    Allele information and '*' symbols are preserved.
    """
    if pd.isna(chain_value) or str(chain_value).strip() == "*":
        return None

    parts = str(chain_value).split(",")

    # Ensure we have at least up to CDR3aa
    if len(parts) < 6:
        return None

    return {
        "chain": chain_type,
        "v_gene": parts[0],
        "d_gene": parts[1],
        "j_gene": parts[2],
        "c_gene": parts[3],
        "cdr3_nt": parts[4],
        "cdr3": parts[5]
    }

Total barcode rows in TRUST4 output: 949


In [5]:
paired_cells = []

for _, row in trust.iterrows():
    # Get the raw barcode string
    barcode_clean = str(row["#barcode"])

    # Parse chain1 as beta (TRB) and chain2 as alpha (TRA)
    beta = parse_chain(row["chain1"], "TRB")
    alpha = parse_chain(row["chain2"], "TRA")

    # Keep only cells that have both beta and alpha chains
    if beta is None or alpha is None:
        continue

    # A chain is 'complete' if it has valid V, J, and CDR3 sequences
    beta_complete = (
        beta["v_gene"] != "*"
        and beta["j_gene"] != "*"
        and beta["cdr3"] not in ["", "*"]
    )

    alpha_complete = (
        alpha["v_gene"] != "*"
        and alpha["j_gene"] != "*"
        and alpha["cdr3"] not in ["", "*"]
    )

    if not beta_complete or not alpha_complete:
        continue

    # Create unique keys for the beta and alpha chains
    beta_key = f"{beta['v_gene']}_{beta['j_gene']}_{beta['cdr3']}"
    alpha_key = f"{alpha['v_gene']}_{alpha['j_gene']}_{alpha['cdr3']}"

    # Create the paired key (the combination of both chains)
    paired_key = f"TRB={beta_key}|TRA={alpha_key}"

    paired_cells.append({
        "barcode_clean": barcode_clean,
        "paired_key": paired_key,
        "beta": beta,
        "alpha": alpha
    })

paired = pd.DataFrame(paired_cells)

if paired.empty:
    raise ValueError("Could not find any suitable barcodes with complete paired TRA-TRB chains.")

print(f"Number of unique barcodes with complete paired TRA-TRB: {paired['barcode_clean'].nunique()}")

Number of unique barcodes with complete paired TRA-TRB: 224


In [6]:
# Count how many unique barcodes share each paired clonotype
clone_sizes = (
    paired.groupby("paired_key")["barcode_clean"]
    .nunique()
    .reset_index(name="clone_size")
    .sort_values(["clone_size", "paired_key"], ascending=[False, True])
    .reset_index(drop=True)
)

# Generate a Clonotype ID starting from the largest clone
clone_sizes["raw_clonotype_id"] = [f"clonotype{i + 1}" for i in range(clone_sizes.shape[0])]

# Merge the Clonotype ID and clone size back into the main cell dataframe
paired = paired.merge(
    clone_sizes[["paired_key", "raw_clonotype_id", "clone_size"]],
    on="paired_key",
    how="left"
)

In [7]:
# Create a long-format table similar to standard experimental VDJ outputs
output_rows = []

for _, row in paired.iterrows():
    barcode_clean = row["barcode_clean"]
    # Append '-1' to match standard 10x barcode formats if needed
    barcode = f"{barcode_clean}-1"

    beta = row["beta"]
    alpha = row["alpha"]

    # First, append the TRB (beta) chain row
    output_rows.append({
        "barcode": barcode,
        "barcode_clean": barcode_clean,
        "raw_clonotype_id": row["raw_clonotype_id"],
        "chain": "TRB",
        "v_gene": beta["v_gene"],
        "d_gene": beta["d_gene"],
        "j_gene": beta["j_gene"],
        "c_gene": beta["c_gene"],
        "cdr3": beta["cdr3"],
        "cdr3_nt": beta["cdr3_nt"],
        "clone_size": row["clone_size"]
    })

    # Next, append the TRA (alpha) chain row for the same cell
    output_rows.append({
        "barcode": barcode,
        "barcode_clean": barcode_clean,
        "raw_clonotype_id": row["raw_clonotype_id"],
        "chain": "TRA",
        "v_gene": alpha["v_gene"],
        "d_gene": alpha["d_gene"],
        "j_gene": alpha["j_gene"],
        "c_gene": alpha["c_gene"],
        "cdr3": alpha["cdr3"],
        "cdr3_nt": alpha["cdr3_nt"],
        "clone_size": row["clone_size"]
    })

trust_paired = pd.DataFrame(output_rows)

# Extract the numeric part of the clonotype ID for proper sorting
trust_paired["clonotype_number"] = (
    trust_paired["raw_clonotype_id"]
    .str.replace("clonotype", "", regex=False)
    .astype(int)
)

# Order chains: TRB first, then TRA for a given barcode
chain_order = {"TRB": 1, "TRA": 2}
trust_paired["chain_order"] = trust_paired["chain"].map(chain_order)

# Sort the final dataframe:
# 1. Largest clones first
# 2. Clonotype ID number
# 3. Barcode sequence
# 4. TRB then TRA
trust_paired = trust_paired.sort_values(
    ["clone_size", "clonotype_number", "barcode_clean", "chain_order"],
    ascending=[False, True, True, True]
)

# Drop helper columns
trust_paired = trust_paired.drop(columns=["clonotype_number", "chain_order"])

# Save to CSV
trust_paired.to_csv(out_path, index=False)

# Print Summary Statistics
print(f"\nTotal paired clonotypes found: {clone_sizes.shape[0]}")
print("\nTop 20 paired clonotypes:")
print(clone_sizes[["raw_clonotype_id", "clone_size", "paired_key"]].head(20))

print("\nFirst 30 rows of output data:")
display(trust_paired.head(30))

print(f"\nOutput saved successfully to:\n{out_path}")


Total paired clonotypes found: 193

Top 20 paired clonotypes:
   raw_clonotype_id  clone_size  \
0        clonotype1           4   
1        clonotype2           3   
2        clonotype3           3   
3        clonotype4           3   
4        clonotype5           3   
5        clonotype6           3   
6        clonotype7           3   
7        clonotype8           2   
8        clonotype9           2   
9       clonotype10           2   
10      clonotype11           2   
11      clonotype12           2   
12      clonotype13           2   
13      clonotype14           2   
14      clonotype15           2   
15      clonotype16           2   
16      clonotype17           2   
17      clonotype18           2   
18      clonotype19           2   
19      clonotype20           2   

                                           paired_key  
0   TRB=TRBV5-1*01_TRBJ1-1*01_CASSLARDTEAFF|TRA=TR...  
1   TRB=TRBV20-1*01_TRBJ2-7*01_CSARDVRVYEQYF|TRA=T...  
2   TRB=TRBV20-1*02_TRBJ2-1*01_CS

,barcode,barcode_clean,raw_clonotype_id,chain,v_gene,d_gene,j_gene,c_gene,cdr3,cdr3_nt,clone_size
202,ACATCAGAGATATGGT-1,ACATCAGAGATATGGT,clonotype1,TRB,TRBV5-1*01,TRBD1*01,TRBJ1-1*01,*,CASSLARDTEAFF,TGCGCCAGCAGCTTGGCCAGGGACACTGAAGCTTTCTTT,4
203,ACATCAGAGATATGGT-1,ACATCAGAGATATGGT,clonotype1,TRA,TRAV25*01,*,TRAJ52*01,TRAC,CAACNAGGTSYGKLTF,TGTGCAGCTTGTAATGCTGGTGGTACTAGCTATGGAAAGCTGACATTT,4
22,AGCAGCCCATCGGTTA-1,AGCAGCCCATCGGTTA,clonotype1,TRB,TRBV5-1*01,TRBD1*01,TRBJ1-1*01,TRBC1,CASSLARDTEAFF,TGCGCCAGCAGCTTGGCCAGGGACACTGAAGCTTTCTTT,4
23,AGCAGCCCATCGGTTA-1,AGCAGCCCATCGGTTA,clonotype1,TRA,TRAV25*01,*,TRAJ52*01,*,CAACNAGGTSYGKLTF,TGTGCAGCTTGTAATGCTGGTGGTACTAGCTATGGAAAGCTGACATTT,4
42,CAGAATCCACTTAACG-1,CAGAATCCACTTAACG,clonotype1,TRB,TRBV5-1*01,TRBD1*01,TRBJ1-1*01,TRBC1,CASSLARDTEAFF,TGCGCCAGCAGCTTGGCCAGGGACACTGAAGCTTTCTTT,4
43,CAGAATCCACTTAACG-1,CAGAATCCACTTAACG,clonotype1,TRA,TRAV25*01,*,TRAJ52*01,TRAC,CAACNAGGTSYGKLTF,TGTGCAGCTTGTAATGCTGGTGGTACTAGCTATGGAAAGCTGACATTT,4
410,TACCTTATCCGTTGTC-1,TACCTTATCCGTTGTC,clonotype1,TRB,TRBV5-1*01,TRBD1*01,TRBJ1-1*01,TRBC1,CASSLARDTEAFF,TGCGCCAGCAGCTTGGCCAGGGACACTGAAGCTTTCTTT,4
411,TACCTTATCCGTTGTC-1,TACCTTATCCGTTGTC,clonotype1,TRA,TRAV25*01,*,TRAJ52*01,*,CAACNAGGTSYGKLTF,TGTGCAGCTTGTAATGCTGGTGGTACTAGCTATGGAAAGCTGACATTT,4
430,CAGAGAGTCAAGGCTT-1,CAGAGAGTCAAGGCTT,clonotype2,TRB,TRBV20-1*01,TRBD1*01,TRBJ2-7*01,TRBC2,CSARDVRVYEQYF,TGCAGTGCTAGAGATGTCAGGGTCTACGAGCAGTACTTC,3
431,CAGAGAGTCAAGGCTT-1,CAGAGAGTCAAGGCTT,clonotype2,TRA,TRAV26-1*01,*,TRAJ22*01,TRAC,CIVRVAGSARQLTF,TGCATCGTCAGAGTCGCCGGTTCTGCAAGGCAACTGACCTTT,3



Output saved successfully to:
/home/pc11/TRUST4_workspace/trust4_paired_all_clonotypes.csv


In [8]:
import pandas as pd

# Load TRUST4 barcode report
trust4_report = pd.read_csv(
    "/home/pc11/TRUST4_workspace/SRR13342054_TRUST4_barcode_report.tsv",
    sep="\t"
)

print(f"📊 Total cells with TCR/BCR: {len(trust4_report)}")
print(f"📊 Columns: {list(trust4_report.columns)}")
print(f"\n🔍 First 5 rows:")
trust4_report.head()

📊 Total cells with TCR/BCR: 949
📊 Columns: ['#barcode', 'cell_type', 'chain1', 'chain2', 'secondary_chain1', 'secondary_chain2']

🔍 First 5 rows:


,#barcode,cell_type,chain1,chain2,secondary_chain1,secondary_chain2
0,GACGTGCAGGCGATAC,abT,"TRBV7-3*01,TRBD2*01,TRBJ1-2*01,TRBC1,TGTGCCAGC...","TRAV4*01,*,TRAJ42*01,*,GGCCTCGTGGGTGATTATGTAGG...","TRBV29-1*01,TRBD2*02,TRBJ2-3*01,*,TGCAGCGTTGAA...","TRAV9-2*03,*,TRAJ32*02,*,TGTGCTCTGACCGCGGGGGTG..."
1,CAGCTGGCAAGCGATG,abT,"TRBV5-5*02,TRBD1*01,TRBJ1-1*01,TRBC1,TGTGCCAGC...",*,*,*
2,TGCGTGGAGTACGATA,abT,"TRBV7-3*01,*,TRBJ2-1*01,*,TGTGCCAGCAGGCCTGAAGG...",*,*,*
3,AGTGGGAGTGAGGCTA,abT,"TRBV2*01,TRBD1*01,TRBJ1-1*01,*,TGTGCCAGCAGATTA...",*,*,*
4,CATCGAAAGACACGAC,abT,*,"TRAV26-2*01,*,TRAJ29*01,*,TGCATCCTGAGACGGAATTC...",*,*


In [9]:
# Count cell types detected by TRUST4
if 'cell_type' in trust4_report.columns:
    print("🧬 Cell type distribution from TRUST4:")
    print(trust4_report['cell_type'].value_counts())
    print(f"\n📊 Total cells with immune receptors: {len(trust4_report)}")

🧬 Cell type distribution from TRUST4:
cell_type
abT    918
B       22
gdT      9
Name: count, dtype: int64

📊 Total cells with immune receptors: 949
